# Part 2 — NumPy & Pandas Data Wrangling 

# NumPy arrays: video metrics

Load the `views`, `likes`, `dislikes`, and `comment_count` columns from the GB, IN, and US CSV files, in that order. Missing or invalid metric values are replaced with `0` before converting to NumPy integer arrays. Each one-dimensional array uses the same row order, so an index refers to the same record across all four arrays.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

country_codes = ("GB", "IN", "US")
expected_files = tuple(f"{country}videos.csv" for country in country_codes)
possible_data_dirs = (
    Path.cwd(),
    *(parent / "part2_numpy_pandas" for parent in (Path.cwd(), *Path.cwd().parents)),
)
DATA_DIR = next(
    (
        directory
        for directory in possible_data_dirs
        if all((directory / filename).is_file() for filename in expected_files)
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find GBvideos.csv, INvideos.csv, and USvideos.csv "
        "in the current directory or a part2_numpy_pandas folder."
    )

metric_columns = ("views", "likes", "dislikes", "comment_count")
country_frames = []
for filename in expected_files:
    country_frame = pd.read_csv(DATA_DIR / filename, usecols=metric_columns)
    for column in metric_columns:
        country_frame[column] = (
            pd.to_numeric(country_frame[column], errors="coerce")
            .fillna(0)
            .astype(np.int64)
        )
    country_frames.append(country_frame)

video_metrics = pd.concat(country_frames, ignore_index=True)
views = video_metrics["views"].to_numpy(dtype=np.int64)
likes = video_metrics["likes"].to_numpy(dtype=np.int64)
dislikes = video_metrics["dislikes"].to_numpy(dtype=np.int64)
comment_count = video_metrics["comment_count"].to_numpy(dtype=np.int64)
metric_array = video_metrics.to_numpy(dtype=np.int64)

for name, array in (
    ("views", views),
    ("likes", likes),
    ("dislikes", dislikes),
    ("comment_count", comment_count),
):
    print(f"{name}: shape={array.shape}, dtype={array.dtype}")
print(f"Combined records: {len(video_metrics):,}")

views: shape=(46230,), dtype=int64
likes: shape=(46230,), dtype=int64
dislikes: shape=(46230,), dtype=int64
comment_count: shape=(46230,), dtype=int64
Combined records: 46,230


## Indexing and slicing

`np.sort(views)[-10:][::-1]` selects and orders the ten largest view counts. `metric_array[::100]` selects every hundredth row from the four-column array; the displayed rows include their original row indices. The final example uses row and column slices to select five consecutive records and all four metrics.

In [2]:
largest_10_views = np.sort(views)[-10:][::-1]
print("10 largest view counts:")
print(largest_10_views)
every_100th_record = metric_array[::100]

every_100th_indices = np.arange(0, len(metric_array), 100)
indexed_sample = np.column_stack((every_100th_indices, every_100th_record))
print(f"\nEvery 100th record (row index, views, likes, dislikes, comment_count); shape={every_100th_record.shape}:")
print(indexed_sample.tolist())

metric_subarray = metric_array[10:15, :]
print("\nChosen sub-array: rows 10 through 14, all four metrics:")
print(metric_subarray)

10 largest view counts:
[360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]

Every 100th record (row index, views, likes, dislikes, comment_count); shape=(463, 4):
[[0, 84344, 2155, 163, 475], [100, 656734, 31256, 3627, 2876], [200, 109438, 2796, 212, 617], [300, 289683, 12026, 324, 317], [400, 124665, 3186, 242, 703], [500, 67649, 3445, 189, 0], [600, 156685, 4004, 304, 883], [700, 79924, 1416, 120, 238], [800, 195161, 4987, 379, 1100], [900, 9083642, 421644, 10192, 7843], [1000, 207338, 12905, 2398, 811], [1100, 378844, 18415, 820, 1634], [1200, 259556, 7098, 397, 698], [1300, 193359, 5144, 177, 359], [1400, 9714, 332, 13, 36], [1500, 216727, 9105, 949, 196], [1600, 33827, 1074, 34, 131], [1700, 429799, 10908, 490, 807], [1800, 340953, 11365, 1188, 2291], [1900, 90008, 2153, 84, 178], [2000, 141471, 4364, 259, 485], [2100, 84232, 2576, 64, 547], [2200, 421423, 11715, 1038, 1139], [2300, 49866, 1011, 90, 208], [2400, 186414, 4899, 330

## Engagement score without loops

The like ratio is likes divided by total likes plus dislikes; the comment rate is comments per view. Both use `np.divide` with a zero-filled output and a `where` mask, so records with zero denominators receive `0` rather than a division error. Min-max scaling puts views, likes, and comments on a common 0–1 scale; a constant-valued metric maps to zeros. The engagement score weights normalized views at 20%, normalized likes at 50%, and normalized comments at 30%. Likes and comments receive most of the weight because they represent active responses, while views provide a smaller reach component. `np.log1p` compresses the highly skewed view-count distribution, making differences among low- and mid-view records easier to see on a plot. The final boolean mask uses the 90th-percentile score cutoff; ties at the cutoff can flag slightly more than 10% of records.

In [3]:
like_ratio = np.divide(
    likes,
    likes + dislikes,
    out=np.zeros(likes.shape, dtype=np.float64),
    where=(likes + dislikes) != 0,
)
comment_rate = np.divide(
    comment_count,
    views,
    out=np.zeros(views.shape, dtype=np.float64),
    where=views != 0,
)

def min_max_normalize(values):
    value_range = values.max() - values.min()
    return np.divide(
        values - values.min(),
        value_range,
        out=np.zeros(values.shape, dtype=np.float64),
        where=value_range != 0,
    )

views_normalized = min_max_normalize(views)
likes_normalized = min_max_normalize(likes)
comments_normalized = min_max_normalize(comment_count)

engagement_score = (
    0.20 * views_normalized
    + 0.50 * likes_normalized
    + 0.30 * comments_normalized
)
log_views = np.log1p(views)

top_10_percent_cutoff = np.percentile(engagement_score, 90)
top_10_percent_mask = engagement_score >= top_10_percent_cutoff
top_record_indices = np.flatnonzero(top_10_percent_mask)

print("Like ratio (first 5):", like_ratio[:5])
print("Comment rate (first 5):", comment_rate[:5])
print("Log-scaled views (first 5):", log_views[:5])
print("Engagement score (first 5):", engagement_score[:5])
print(
    f"Top-decile score cutoff: {top_10_percent_cutoff:.6f}; "
    f"flagged records: {len(top_record_indices):,} / {len(engagement_score):,}"
)
print("Top-record indices (first 10):", top_record_indices[:10])

Like ratio (first 5): [0.92968076 0.94252874 0.92654028 0.89732143 0.9503866 ]
Comment rate (first 5): [0.0056317  0.00739474 0.00408733 0.00834079 0.0010094 ]
Log-scaled views (first 5): [11.34267081 12.34772411 12.73332166 11.98851969 10.99275633]
Engagement score (first 5): [2.68366392e-04 8.82163586e-04 1.02632919e-03 6.82469098e-04
 8.22683191e-05]
Top-decile score cutoff: 0.005097; flagged records: 4,623 / 46,230
Top-record indices (first 10): [302 369 399 405 479 533 534 545 566 589]


## Reshape and broadcast daily country counts

Count how many videos are trending on each date in each country. The matrix columns follow `country_codes` (`GB`, `IN`, `US`), and the flattened values are reshaped back into a days-by-countries matrix. Broadcasting divides each row by its daily total to find each country's share of that day's trending videos. It also divides each country column by that country's average daily count; a value of `1` means the day is exactly at its country average. Zero denominators produce zeros.

In [4]:
daily_date_records = pd.concat(
    (
        pd.read_csv(DATA_DIR / "GBvideos.csv", usecols=("trending_date",)).assign(country_code="GB"),
        pd.read_csv(DATA_DIR / "INvideos.csv", usecols=("trending_date",)).assign(country_code="IN"),
        pd.read_csv(DATA_DIR / "USvideos.csv", usecols=("trending_date",)).assign(country_code="US"),
    ),
    ignore_index=True,
)
daily_date_records["trending_date"] = pd.to_datetime(
    daily_date_records["trending_date"], format="%y.%d.%m"
)
daily_counts_table = (
    daily_date_records.groupby(["trending_date", "country_code"])
    .size()
    .unstack("country_code", fill_value=0)
    .reindex(columns=country_codes, fill_value=0)
    .sort_index()
)

daily_dates = daily_counts_table.index
daily_counts_flat = daily_counts_table.to_numpy(dtype=np.int64).flatten()
daily_trending_counts = daily_counts_flat.reshape(len(daily_dates), len(country_codes))

daily_totals = daily_trending_counts.sum(axis=1, keepdims=True)
daily_country_shares = np.divide(
    daily_trending_counts,
    daily_totals,
    out=np.zeros(daily_trending_counts.shape, dtype=np.float64),
    where=daily_totals != 0,
)

country_daily_averages = daily_trending_counts.mean(axis=0, keepdims=True)
daily_counts_vs_country_average = np.divide(
    daily_trending_counts,
    country_daily_averages,
    out=np.zeros(daily_trending_counts.shape, dtype=np.float64),
    where=country_daily_averages != 0,
)

print(f"Daily count matrix shape (days, countries): {daily_trending_counts.shape}")
print("Country column order:", country_codes)
print("Flattened array shape:", daily_counts_flat.shape)
print("First 5 daily counts:\n", daily_trending_counts[:5])
print("First 5 daily country shares:\n", np.round(daily_country_shares[:5], 3))
print(
    "First 5 days as multiples of each country's average:\n",
    np.round(daily_counts_vs_country_average[:5], 3),
)

Daily count matrix shape (days, countries): (80, 3)
Country column order: ('GB', 'IN', 'US')
Flattened array shape: (240,)
First 5 daily counts:
 [[200 200 200]
 [200 201 202]
 [200 200 200]
 [200 201 201]
 [201 203 200]]
First 5 daily country shares:
 [[0.333 0.333 0.333]
 [0.332 0.333 0.335]
 [0.333 0.333 0.333]
 [0.332 0.334 0.334]
 [0.333 0.336 0.331]]
First 5 days as multiples of each country's average:
 [[1.061 1.061 0.995]
 [1.061 1.067 1.005]
 [1.061 1.061 0.995]
 [1.061 1.067 1.   ]
 [1.067 1.077 0.995]]


## Load, inspect, and clean the country data

Read the full GB, IN, and US files into separate DataFrames and add a `country` column. The quality report below measures issues before cleanup. Exact duplicate `(country, video_id, trending_date)` rows are redundant daily observations, so only extra copies are removed; the same video appearing on different dates is expected and retained. The `trending_date` strings use the `%y.%d.%m` format and `publish_time` is parsed as UTC. Numeric columns use nullable integer types so missing values are not mistaken for zero; the earlier NumPy exercise intentionally fills missing metrics with zero only for its arrays. `[none]` in `tags` is treated as a missing value; missing descriptions are preserved. Error/removal and disabled-comment/rating flags are retained for downstream filtering rather than silently dropping those records.

The raw files have no invalid date strings, no trending dates before the publish calendar date, no invalid or negative numeric values, and no blank video IDs, titles, or channel names. The report lists each issue and count for every country and the uncleaned combined data. Repeated `video_id` values alone are not considered duplicates because a video can trend on many dates.

In [5]:
text_columns = (
    "video_id", "title", "channel_title", "tags", "thumbnail_link", "description"
)
numeric_columns = ("category_id", "views", "likes", "dislikes", "comment_count")
metric_columns = ("views", "likes", "dislikes", "comment_count")
flag_columns = ("comments_disabled", "ratings_disabled", "video_error_or_removed")

raw_country_frames = {}
for country_code, filename in zip(country_codes, expected_files):
    country_frame = pd.read_csv(DATA_DIR / filename)
    country_frame["country"] = country_code
    raw_country_frames[country_code] = country_frame

def count_quality_issues(frame):
    numeric_values = frame.loc[:, numeric_columns].apply(pd.to_numeric, errors="coerce")
    trending_dates = pd.to_datetime(
        frame["trending_date"], format="%y.%d.%m", errors="coerce"
    )
    publish_times = pd.to_datetime(frame["publish_time"], errors="coerce", utc=True)
    publish_days = publish_times.dt.tz_convert(None).dt.normalize()
    duplicate_key = ["video_id", "trending_date"]
    if "country" in frame.columns:
        duplicate_key.insert(0, "country")
    return {
        "rows": len(frame),
        "missing_likes": int(frame["likes"].isna().sum()),
        "missing_comment_count": int(frame["comment_count"].isna().sum()),
        "missing_description": int(frame["description"].isna().sum()),
        "tags_[none]": int(
            frame["tags"].astype("string").str.strip().str.lower().eq("[none]").sum()
        ),
        "duplicate_video_date_rows": int(frame.duplicated(duplicate_key).sum()),
        "video_error_or_removed": int(frame["video_error_or_removed"].sum()),
        "comments_disabled": int(frame["comments_disabled"].sum()),
        "ratings_disabled": int(frame["ratings_disabled"].sum()),
        "invalid_date_values": int(
            (trending_dates.isna() | publish_times.isna()).sum()
        ),
        "trending_before_publish_day": int((trending_dates < publish_days).sum()),
        "invalid_numeric_values": int(
            sum(
                (numeric_values[column].isna() & frame[column].notna()).sum()
                for column in numeric_columns
            )
        ),
        "negative_metric_values": int((numeric_values.loc[:, metric_columns] < 0).sum().sum()),
        "missing_or_blank_required_text": int(
            sum(
                (
                    frame[column].isna()
                    | frame[column].astype("string").str.strip().eq("")
                ).sum()
                for column in ("video_id", "title", "channel_title")
            )
        ),
    }

raw_combined = pd.concat(raw_country_frames.values(), ignore_index=True)
quality_report = pd.DataFrame.from_dict(
    {
        **{
            country_code: count_quality_issues(frame)
            for country_code, frame in raw_country_frames.items()
        },
        "combined (before cleanup)": count_quality_issues(raw_combined),
    },
    orient="index",
)
print("Data-quality issue counts (before cleanup):")
display(quality_report)

country_frames = {}
for country_code, raw_frame in raw_country_frames.items():
    country_frame = raw_frame.copy()
    country_frame["trending_date"] = pd.to_datetime(
        country_frame["trending_date"], format="%y.%d.%m"
    )
    country_frame["publish_time"] = pd.to_datetime(
        country_frame["publish_time"], utc=True
    )
    for column in numeric_columns:
        country_frame[column] = pd.to_numeric(
            country_frame[column], errors="raise"
        ).astype("Int64")
    for column in text_columns:
        country_frame[column] = country_frame[column].astype("string")
    country_frame["tags"] = country_frame["tags"].mask(
        country_frame["tags"].str.strip().str.lower().eq("[none]"), pd.NA
    )
    for column in flag_columns:
        country_frame[column] = country_frame[column].astype("boolean")
    country_frame["country"] = pd.Categorical(
        country_frame["country"], categories=country_codes
    )
    country_frames[country_code] = country_frame.drop_duplicates(
        subset=["video_id", "trending_date"], keep="first"
    ).reset_index(drop=True)

combined_data = pd.concat(country_frames.values(), ignore_index=True)
inspection_frames = [
    *country_frames.items(),
    ("combined", combined_data),
]
for frame_name, frame in inspection_frames:
    print(f"\n{'=' * 20} {frame_name} {'=' * 20}")
    print("shape:", frame.shape)
    print("head():")
    display(frame.head())
    print("info():")
    frame.info()
    print("describe(include='all').T:")
    display(frame.describe(include="all").T)

Data-quality issue counts (before cleanup):


,rows,missing_likes,missing_comment_count,missing_description,tags_[none],duplicate_video_date_rows,video_error_or_removed,comments_disabled,ratings_disabled,invalid_date_values,trending_before_publish_day,invalid_numeric_values,negative_metric_values,missing_or_blank_required_text
GB,15075,4,12,468,1412,75,30,374,203,0,0,0,0,0
IN,15075,4,12,592,1209,75,30,345,172,0,0,0,0,0
US,16080,4,12,564,1164,80,32,329,169,0,0,0,0,0
combined (before cleanup),46230,12,36,1624,3785,230,92,1048,544,0,0,0,0,0



==================== GB ====================
shape: (15000, 17)
head():


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,2017-11-14,Wild Winter Prank,OfficialRecipe78,28,2017-10-18 07:42:02+00:00,<NA>,84344,2155,163,475,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
1,FoQAEGxMMQD,2017-11-14,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14 12:56:18+00:00,Minecraft|Paris,230434,5412,330,1704,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
2,wHqDBuStXq3,2017-11-14,Epic Space Challenge,UltimateReview54,23,2017-11-13 06:19:10+00:00,Paris,338852,16813,1333,1385,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
3,ZId0IGzSaaZ,2017-11-14,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14 11:37:33+00:00,Phones,160896,4221,483,1342,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
4,1-xA4eRrYSH,2017-11-14,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14 03:13:26+00:00,Chess|Minecraft|Festival|Guitar,59441,1475,77,60,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB


info():
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   video_id                15000 non-null  string             
 1   trending_date           15000 non-null  datetime64[us]     
 2   title                   15000 non-null  string             
 3   channel_title           15000 non-null  string             
 4   category_id             15000 non-null  Int64              
 5   publish_time            15000 non-null  datetime64[us, UTC]
 6   tags                    13591 non-null  string             
 7   views                   15000 non-null  Int64              
 8   likes                   14996 non-null  Int64              
 9   dislikes                15000 non-null  Int64              
 10  comment_count           14988 non-null  Int64              
 11  thumbnail_link          15000 non-null  stri

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
video_id,15000,4113,EZaG9_2Yi1u,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,15000,NaN,NaN,NaN,2017-12-21 00:00:00,2017-11-14 00:00:00,2017-12-02 00:00:00,2017-12-21 00:00:00,2018-01-09 00:00:00,2018-01-27 00:00:00,NaN
title,15000,3629,Kairo - Chess (Official Video),44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,15000,387,QuietChallenge75,1219,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,15000.0,<NA>,<NA>,<NA>,17.098067,1.0,10.0,20.0,24.0,28.0,7.750419
publish_time,15000,NaN,NaN,NaN,2017-12-15 18:33:05.668800+00:00,2017-10-09 04:32:45+00:00,2017-11-27 02:33:26+00:00,2017-12-15 09:10:57+00:00,2018-01-03 09:51:16+00:00,2018-01-27 21:45:00+00:00,NaN
tags,13591,2146,Minecraft,269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,15000.0,<NA>,<NA>,<NA>,1584041.316667,1066.0,121112.75,317014.0,867575.75,360760164.0,9127636.529875
likes,14996.0,<NA>,<NA>,<NA>,65266.896439,0.0,3652.75,10540.0,31804.5,27537822.0,478104.97395
dislikes,15000.0,<NA>,<NA>,<NA>,2943.512133,0.0,194.0,569.0,1691.0,618175.0,15442.985713


... (8 outputs left out)


==================== combined ====================
shape: (46000, 17)
head():


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,2017-11-14,Wild Winter Prank,OfficialRecipe78,28,2017-10-18 07:42:02+00:00,<NA>,84344,2155,163,475,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
1,FoQAEGxMMQD,2017-11-14,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14 12:56:18+00:00,Minecraft|Paris,230434,5412,330,1704,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
2,wHqDBuStXq3,2017-11-14,Epic Space Challenge,UltimateReview54,23,2017-11-13 06:19:10+00:00,Paris,338852,16813,1333,1385,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
3,ZId0IGzSaaZ,2017-11-14,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14 11:37:33+00:00,Phones,160896,4221,483,1342,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
4,1-xA4eRrYSH,2017-11-14,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14 03:13:26+00:00,Chess|Minecraft|Festival|Guitar,59441,1475,77,60,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB


info():
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   video_id                46000 non-null  string             
 1   trending_date           46000 non-null  datetime64[us]     
 2   title                   46000 non-null  string             
 3   channel_title           46000 non-null  string             
 4   category_id             46000 non-null  Int64              
 5   publish_time            46000 non-null  datetime64[us, UTC]
 6   tags                    42226 non-null  string             
 7   views                   46000 non-null  Int64              
 8   likes                   45988 non-null  Int64              
 9   dislikes                46000 non-null  Int64              
 10  comment_count           45964 non-null  Int64              
 11  thumbnail_link          46000 non-null  stri

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
video_id,46000,14699,iAvRHGeUpBR,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trending_date,46000,NaN,NaN,NaN,2017-12-21 20:52:10.434782,2017-11-14 00:00:00,2017-12-03 00:00:00,2017-12-22 00:00:00,2018-01-10 00:00:00,2018-02-01 00:00:00,NaN
title,46000,10404,Sunny Park - Bangalore (Live),66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,46000,438,QuietChallenge75,2451,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_id,46000.0,<NA>,<NA>,<NA>,18.644065,1.0,10.0,22.0,24.0,28.0,7.647781
publish_time,46000,NaN,NaN,NaN,2017-12-17 06:39:51.698956+00:00,2017-10-09 04:32:45+00:00,2017-11-27 14:29:23+00:00,2017-12-17 04:58:09+00:00,2018-01-05 17:17:40+00:00,2018-02-01 21:56:14+00:00,NaN
tags,42226,6055,Minecraft,788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
views,46000.0,<NA>,<NA>,<NA>,1179665.244652,1066.0,129453.75,318389.0,824049.5,360760164.0,5926282.241972
likes,45988.0,<NA>,<NA>,<NA>,45850.674676,0.0,3654.0,10211.0,28987.25,27537822.0,297593.992752
dislikes,46000.0,<NA>,<NA>,<NA>,2329.11787,0.0,213.0,595.0,1667.0,618175.0,10541.533689


## Filter and sort records

The filters use the cleaned, deduplicated `combined_data`. The first mask uses `&` to find records with at least 1,000,000 views and more than 50,000 likes. The engagement score follows the earlier 20% views, 50% likes, and 30% comments weighting after min-max scaling; missing metric values are treated as zero for scoring. The category filter uses `|` for Music or Gaming, combined with `&` for views strictly above the overall median. Full matches remain in `million_views_likes` and `music_or_gaming_above_median`; their tables show a sorted 20-record preview to keep output readable. `top_20_engagement` contains the complete top 20.

In [6]:
filtering_data = combined_data.copy()
category_names = pd.read_csv(DATA_DIR / "categories.csv")
category_lookup = category_names.set_index("category_id")["category_name"]
filtering_data["category_name"] = filtering_data["category_id"].map(category_lookup)

metric_values = filtering_data.loc[:, ["views", "likes", "comment_count"]].fillna(0)
metric_values = metric_values.to_numpy(dtype=np.float64)
metric_min = metric_values.min(axis=0)
metric_range = metric_values.max(axis=0) - metric_min
normalized_metrics = np.divide(
    metric_values - metric_min,
    metric_range,
    out=np.zeros(metric_values.shape, dtype=np.float64),
    where=metric_range != 0,
)
filtering_data["engagement_score"] = (
    0.20 * normalized_metrics[:, 0]
    + 0.50 * normalized_metrics[:, 1]
    + 0.30 * normalized_metrics[:, 2]
)

readable_columns = [
    "country", "video_id", "trending_date", "title", "category_name",
    "views", "likes", "engagement_score",
]

million_views_mask = filtering_data["views"] >= 1_000_000
high_likes_mask = filtering_data["likes"].fillna(0) > 50_000
million_views_likes = (
    filtering_data.loc[million_views_mask & high_likes_mask, readable_columns]
    .sort_values("views", ascending=False, kind="stable")
)
print(f"(a) 1M+ views and >50K likes: {len(million_views_likes):,} records")
display(million_views_likes.loc[:, readable_columns].iloc[:20])

top_20_engagement = (
    filtering_data.sort_values(
        "engagement_score", ascending=False, kind="stable"
    ).iloc[:20]
)
print("(b) Top 20 records by engagement score:")
display(top_20_engagement.loc[:, readable_columns])

overall_median_views = filtering_data["views"].median()
music_or_gaming_mask = (
    (filtering_data["category_name"] == "Music")
    | (filtering_data["category_name"] == "Gaming")
)
above_median_views_mask = filtering_data["views"] > overall_median_views
music_or_gaming_above_median = (
    filtering_data.loc[
        music_or_gaming_mask & above_median_views_mask, readable_columns
    ]
    .sort_values("views", ascending=False, kind="stable")
)
print(
    f"(c) Music or Gaming and views > overall median "
    f"({overall_median_views:,.0f}): "
    f"{len(music_or_gaming_above_median):,} records"
)
display(music_or_gaming_above_median.loc[:, readable_columns].iloc[:20])

(a) 1M+ views and >50K likes: 6,718 records


,country,video_id,trending_date,title,category_name,views,likes,engagement_score
10400,GB,RdqSDDng84n,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.930707
14800,GB,QYwpwP6y0Eb,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.619904
10200,GB,RdqSDDng84n,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.739680
14601,GB,QYwpwP6y0Eb,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.510684
8000,GB,WQEWsMmimzZ,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.648808
10000,GB,RdqSDDng84n,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.609555
14401,GB,QYwpwP6y0Eb,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.416591
9800,GB,RdqSDDng84n,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.543696
7800,GB,WQEWsMmimzZ,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.509274
14201,GB,QYwpwP6y0Eb,2018-01-24,The Hollow Pines - Phones (Official Video) WATCH,Music,181785943,5073905,0.328257


(b) Top 20 records by engagement score:


,country,video_id,trending_date,title,category_name,views,likes,engagement_score
10400,GB,RdqSDDng84n,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.930707
10200,GB,RdqSDDng84n,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.739680
8000,GB,WQEWsMmimzZ,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.648808
14800,GB,QYwpwP6y0Eb,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.619904
10000,GB,RdqSDDng84n,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.609555
9800,GB,RdqSDDng84n,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.543696
14601,GB,QYwpwP6y0Eb,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.510684
7800,GB,WQEWsMmimzZ,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.509274
9600,GB,RdqSDDng84n,2018-01-01,Lena Ortiz - Monsoon (Live),Music,169181540,12914095,0.436462
14401,GB,QYwpwP6y0Eb,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.416591


(c) Music or Gaming and views > overall median (318,389): 9,126 records


,country,video_id,trending_date,title,category_name,views,likes,engagement_score
10400,GB,RdqSDDng84n,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.930707
14800,GB,QYwpwP6y0Eb,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.619904
10200,GB,RdqSDDng84n,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.739680
14601,GB,QYwpwP6y0Eb,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.510684
8000,GB,WQEWsMmimzZ,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.648808
10000,GB,RdqSDDng84n,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.609555
14401,GB,QYwpwP6y0Eb,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.416591
9800,GB,RdqSDDng84n,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.543696
7800,GB,WQEWsMmimzZ,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.509274
14201,GB,QYwpwP6y0Eb,2018-01-24,The Hollow Pines - Phones (Official Video) WATCH,Music,181785943,5073905,0.328257


## Missing values and duplicate records

The table reports missing counts and percentages on the cleaned, deduplicated combined data. Missing `likes` and `comment_count` are retained as nullable values in the source data because missing does not prove zero activity; `fillna(0)` is applied only to a separate analysis copy when a numeric calculation needs an explicit value. Missing descriptions are retained because they are optional text. The literal `[none]` in tags means no tags were supplied, so it was converted to a missing value during cleaning and remains missing rather than being dropped.

A duplicate is a repeated `(country, video_id, trending_date)` key: one video appearing on different dates is expected, and the same ID in a different country is a separate observation. `drop_duplicates(..., keep="first")` removes only extra records with that key. The 46,230 raw rows contain 230 extra duplicate rows (75 GB, 75 IN, 80 US), leaving 46,000. `dropna()` is demonstrated only on required identity/date fields; no rows are lost because those fields are complete.

In [7]:
missing_counts = combined_data.isna().sum()
missing_percentages = (missing_counts / len(combined_data) * 100).round(4)
missing_summary = pd.DataFrame(
    {
        "missing_count": missing_counts,
        "missing_percent": missing_percentages,
    }
).loc[lambda table: table["missing_count"] > 0]
print("Columns with missing values in the cleaned combined data:")
display(missing_summary)

required_fields = ["country", "video_id", "trending_date"]
records_before_required_dropna = len(combined_data)
records_with_required_fields = combined_data.dropna(subset=required_fields)
required_rows_dropped = records_before_required_dropna - len(records_with_required_fields)
print(
    f"dropna(subset={required_fields}) removed "
    f"{required_rows_dropped:,} rows; "
    f"{len(records_with_required_fields):,} remain."
)

duplicate_key = ["country", "video_id", "trending_date"]
raw_duplicate_rows = int(raw_combined.duplicated(duplicate_key).sum())
deduplicated_raw = raw_combined.drop_duplicates(
    subset=duplicate_key, keep="first"
)
print(
    f"drop_duplicates(subset={duplicate_key}) removed "
    f"{raw_duplicate_rows:,} rows from {len(raw_combined):,}; "
    f"{len(deduplicated_raw):,} remain."
)

analysis_metrics = combined_data.loc[:, ["likes", "comment_count"]].copy()
missing_metrics_filled = int(analysis_metrics.isna().sum().sum())
analysis_metrics = analysis_metrics.fillna(0)
print(
    f"fillna(0) filled {missing_metrics_filled:,} missing like/comment values "
    "in this analysis copy only; combined_data remains unchanged."
)

Columns with missing values in the cleaned combined data:


,missing_count,missing_percent
tags,3774,8.2043
likes,12,0.0261
comment_count,36,0.0783
description,1615,3.5109


dropna(subset=['country', 'video_id', 'trending_date']) removed 0 rows; 46,000 remain.
drop_duplicates(subset=['country', 'video_id', 'trending_date']) removed 230 rows from 46,230; 46,000 remain.
fillna(0) filled 48 missing like/comment values in this analysis copy only; combined_data remains unchanged.


## Join category names to trending records

Load the category lookup and merge it to the cleaned combined records on `category_id`. A left merge keeps every trending record even if a category ID is not in the lookup; an inner merge could silently discard such records. `validate="many_to_one"` checks that the lookup has at most one name per category ID. The row-count and merge-status checks below confirm whether all records received a category match.

In [8]:
category_lookup = pd.read_csv(DATA_DIR / "categories.csv")
rows_before_category_merge = len(combined_data)
video_data_with_categories = combined_data.merge(
    category_lookup,
    on="category_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
rows_after_category_merge = len(video_data_with_categories)
category_merge_counts = video_data_with_categories["_merge"].value_counts()
unmatched_category_rows = int(
    category_merge_counts.get("left_only", 0)
)

print("Merge type: left")
print(f"Rows before merge: {rows_before_category_merge:,}")
print(f"Rows after merge: {rows_after_category_merge:,}")
print(f"Unmatched category rows: {unmatched_category_rows:,}")
print("Merge status counts:")
display(category_merge_counts)
assert rows_after_category_merge == rows_before_category_merge
assert unmatched_category_rows == 0

video_data_with_categories = video_data_with_categories.drop(columns="_merge")
display(
    video_data_with_categories[
        ["country", "video_id", "category_id", "category_name", "views"]
    ].head()
)

Merge type: left
Rows before merge: 46,000
Rows after merge: 46,000
Unmatched category rows: 0
Merge status counts:


_merge
both          46000
left_only         0
right_only        0
Name: count, dtype: int64

,country,video_id,category_id,category_name,views
0,GB,u_nKn025TZS,28,Science & Technology,84344
1,GB,FoQAEGxMMQD,26,Howto & Style,230434
2,GB,wHqDBuStXq3,23,Comedy,338852
3,GB,ZId0IGzSaaZ,24,Entertainment,160896
4,GB,1-xA4eRrYSH,10,Music,59441


## Aggregate engagement and publishing patterns

Summarize record count and average views, likes, and engagement score separately by category and country. Category frequencies use `value_counts()` and are shown as both counts and percentages. Publishing-day and publishing-hour summaries use the mean views per record to describe when videos tend to have higher view counts; `publish_time` is UTC, so these time buckets are UTC. Means can be influenced by viral outliers, and describe this dataset rather than proving that publish time causes higher views.

In [9]:
aggregation_data = video_data_with_categories.copy()
aggregation_metrics = aggregation_data.loc[
    :, ["views", "likes", "comment_count"]
].fillna(0).to_numpy(dtype=np.float64)
metric_min = aggregation_metrics.min(axis=0)
metric_range = aggregation_metrics.max(axis=0) - metric_min
normalized_metrics = np.divide(
    aggregation_metrics - metric_min,
    metric_range,
    out=np.zeros(aggregation_metrics.shape, dtype=np.float64),
    where=metric_range != 0,
)
aggregation_data["engagement_score"] = (
    0.20 * normalized_metrics[:, 0]
    + 0.50 * normalized_metrics[:, 1]
    + 0.30 * normalized_metrics[:, 2]
)

category_summary = (
    aggregation_data.groupby("category_name", observed=True)
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
        average_likes=("likes", "mean"),
        average_engagement_score=("engagement_score", "mean"),
    )
    .sort_values("record_count", ascending=False)
)
country_summary = (
    aggregation_data.groupby("country", observed=True)
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
        average_likes=("likes", "mean"),
        average_engagement_score=("engagement_score", "mean"),
    )
    .sort_values("record_count", ascending=False)
)
print("By category:")
display(category_summary)
print("By country:")
display(country_summary)

category_counts = aggregation_data["category_name"].value_counts()
category_distribution = pd.DataFrame(
    {
        "record_count": category_counts,
        "percentage": (category_counts / len(aggregation_data) * 100).round(2),
    }
)
print("Category distribution (value_counts and percentage):")
display(category_distribution)

aggregation_data["publish_day"] = aggregation_data["publish_time"].dt.day_name()
aggregation_data["publish_hour_utc"] = aggregation_data["publish_time"].dt.hour
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
]
views_by_publish_day = (
    aggregation_data.groupby("publish_day", observed=True)
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
    )
    .reindex(weekday_order)
    .sort_values("average_views", ascending=False)
)
views_by_publish_hour = (
    aggregation_data.groupby("publish_hour_utc", observed=True)
    .agg(
        record_count=("video_id", "size"),
        average_views=("views", "mean"),
    )
    .sort_values("average_views", ascending=False)
)
print("Average views by publish weekday (highest first):")
display(views_by_publish_day)
print("Average views by publish hour UTC (highest first):")
display(views_by_publish_hour)
print(
    "Highest average-view weekday:", views_by_publish_day["average_views"].idxmax(),
    "| highest average-view hour UTC:", views_by_publish_hour["average_views"].idxmax(),
)

By category:


,record_count,average_views,average_likes,average_engagement_score
category_name,,,,
Entertainment,12270,901025.648737,28939.305617,0.002482
Music,10310,3190950.47711,134966.433838,0.006744
People & Blogs,3804,551234.121188,19835.40878,0.001848
Comedy,3258,603475.664825,28700.824071,0.001794
News & Politics,2422,283794.747729,3113.164463,0.001084
Film & Animation,2297,577974.023944,17191.443622,0.001419
Gaming,2057,509664.939232,20757.359922,0.001724
Sports,1921,374978.163977,10456.75625,0.000818
Howto & Style,1825,299684.774247,10222.452303,0.000702


By country:


,record_count,average_views,average_likes,average_engagement_score
country,,,,
US,16000,1013904.377687,39560.292323,0.002482
GB,15000,1584041.316667,65266.896439,0.003623
IN,15000,952100.764067,33144.305948,0.002364


Category distribution (value_counts and percentage):


,record_count,percentage
category_name,,
Entertainment,12270,26.67
Music,10310,22.41
People & Blogs,3804,8.27
Comedy,3258,7.08
News & Politics,2422,5.27
Film & Animation,2297,4.99
Gaming,2057,4.47
Sports,1921,4.18
Howto & Style,1825,3.97


Average views by publish weekday (highest first):


,record_count,average_views
publish_day,,
Monday,6710,1591746.691803
Friday,6265,1285800.524022
Tuesday,7102,1258619.734723
Thursday,6600,1110096.182576
Wednesday,6547,1101135.539331
Saturday,6493,973051.715386
Sunday,6283,912928.432914


Average views by publish hour UTC (highest first):


,record_count,average_views
publish_hour_utc,,
12,1859,3244442.150619
8,1853,2080883.651916
16,2046,1521319.550342
21,1847,1431313.109908
15,1975,1350460.770633
14,2032,1309131.609252
7,1967,1240033.180478
6,2028,1227165.537968
10,2109,1193301.844476


Highest average-view weekday: Monday | highest average-view hour UTC: 12


## Time to trend and trending lifespan

`days_to_trend` is the number of calendar days from the UTC publish date to the trending date; same-day trends count as zero days. Trending lifespan counts distinct trending dates for each `(country, video_id)` pair, so the same video in two countries is treated as two country-specific lifespans. Duplicate video/date rows were removed earlier. The top-ten list can have tied values at the cutoff; the stable sort keeps the displayed result to ten records.

In [10]:
time_analysis = video_data_with_categories.copy()
publish_day = time_analysis["publish_time"].dt.tz_convert(None).dt.normalize()
trending_day = time_analysis["trending_date"].dt.normalize()
time_analysis["days_to_trend"] = (trending_day - publish_day).dt.days

print("Days-to-trend distribution (calendar days):")
display(time_analysis["days_to_trend"].describe())
print("Days-to-trend frequency:")
display(time_analysis["days_to_trend"].value_counts().sort_index().head(15))

video_lifespans = (
    time_analysis.groupby(["country", "video_id"], observed=True)
    .agg(
        trending_lifespan_days=("trending_date", "nunique"),
        title=("title", "first"),
        category_name=("category_name", "first"),
        first_trending_date=("trending_date", "min"),
        last_trending_date=("trending_date", "max"),
    )
    .reset_index()
)

longest_lived_videos = (
    video_lifespans.sort_values(
        ["trending_lifespan_days", "country", "video_id"],
        ascending=[False, True, True],
        kind="stable",
    )
    .iloc[:10]
)
print("10 longest-lasting videos within country:")
display(longest_lived_videos)

average_lifespan_by_category = (
    video_lifespans.groupby("category_name", observed=True)
    .agg(
        video_country_records=("video_id", "size"),
        average_lifespan_days=("trending_lifespan_days", "mean"),
    )
    .sort_values("average_lifespan_days", ascending=False)
)
print("Average trending lifespan by category:")
display(average_lifespan_by_category)

Days-to-trend distribution (calendar days):


count    46000.000000
mean         5.090457
std          6.243901
min          0.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         47.000000
Name: days_to_trend, dtype: float64

Days-to-trend frequency:


days_to_trend
0     6136
1     7315
2     6625
3     5363
4     4122
5     3136
6     2384
7     1811
8     1390
9     1098
10     864
11     672
12     543
13     460
14     399
Name: count, dtype: int64

10 longest-lasting videos within country:


,country,video_id,trending_lifespan_days,title,category_name,first_trending_date,last_trending_date
950,GB,DYLzrqG-SrK,25,Official Football Update,Entertainment,2017-12-26,2018-01-19
1019,GB,EZaG9_2Yi1u,25,KAIRO - FESTIVAL (LIVE),Music,2017-11-14,2017-12-08
1232,GB,Hh9b0BBsne0,25,Kairo - Phones (Lyric Video)!,Music,2017-12-06,2017-12-30
1432,GB,Kv1WBQp86cP,25,Sunny Park - Cats (Live) MUST,Music,2017-12-21,2018-01-14
1611,GB,NqokCmZykzK,25,Anya Lux - Paris (Lyric Video),Music,2017-11-26,2017-12-20
1630,GB,O6zdCRwWdf1,25,MIRA SOLIS - WINTER (LYRIC VIDEO)?,Music,2017-12-15,2018-01-08
1846,GB,RdqSDDng84n,25,Lena Ortiz - Monsoon (Live),Music,2017-12-12,2018-01-05
2343,GB,Z1aFIGyR5zv,25,Nova Kane - Robots (Official Video),Music,2017-12-29,2018-01-22
2668,GB,cz0dTlZWwSk,25,Final Cars Speech WATCH,Film & Animation,2017-11-22,2017-12-16
2687,GB,dIbJkzpClRM,25,Sunny Park - Robots (Live),Music,2018-01-02,2018-01-26


Average trending lifespan by category:


,video_country_records,average_lifespan_days
category_name,,
Music,1933,5.333678
Entertainment,3467,3.539083
Comedy,1167,2.791774
People & Blogs,1365,2.786813
Film & Animation,850,2.702353
Gaming,778,2.643959
Education,674,2.440653
Howto & Style,759,2.404480
Autos & Vehicles,425,2.369412


## Combine country data and summarize channels

`concat()` stacks the already-cleaned country DataFrames and preserves the country label. The per-video summary has one row per `(country, video_id)`; a video repeated on multiple trending dates counts once, and `max(views)` is used as its final/highest observed view count rather than summing repeated snapshots. The per-channel summary counts those unique videos and sums their per-video view totals. Channel names can occur in more than one country, so country is part of the grouping and merge key. Merging channel metrics back onto per-video rows repeats each channel total for its videos; `drop_duplicates()` then presents one row per channel.

In [11]:
combined_country_data = pd.concat(
    [country_frames[country_code] for country_code in country_codes],
    ignore_index=True,
)
print("Concatenated shape:", combined_country_data.shape)
print("Rows by country:")
display(combined_country_data["country"].value_counts().reindex(country_codes))

per_video_summary = (
    combined_country_data.groupby(
        ["country", "video_id"], observed=True, as_index=False
    )
    .agg(
        channel_title=("channel_title", "first"),
        title=("title", "first"),
        trending_days=("trending_date", "nunique"),
        total_views=("views", "max"),
    )
)

per_channel_summary = (
    per_video_summary.groupby(
        ["country", "channel_title"], observed=True, as_index=False
    )
    .agg(
        trending_video_count=("video_id", "nunique"),
        total_views=("total_views", "sum"),
    )
)

video_rows_with_channel_totals = per_video_summary.merge(
    per_channel_summary,
    on=["country", "channel_title"],
    how="left",
    validate="many_to_one",
    suffixes=("_per_video", "_per_channel"),
)
channel_summary = (
    video_rows_with_channel_totals[
        ["country", "channel_title", "trending_video_count", "total_views_per_channel"]
    ]
    .drop_duplicates()
    .rename(columns={"total_views_per_channel": "total_views"})
    .sort_values("total_views", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print("Per-video summary rows:", len(per_video_summary))
print("Per-channel rows:", len(per_channel_summary))
print("Channel rows after merge and deduplication:", len(channel_summary))
print("Top 10 channels by total views (within country):")
display(channel_summary.head(10))

Concatenated shape: (46000, 17)
Rows by country:


country
GB    15000
IN    15000
US    16000
Name: count, dtype: int64

Per-video summary rows: 15059
Per-channel rows: 1203
Channel rows after merge and deduplication: 1203
Top 10 channels by total views (within country):


,country,channel_title,trending_video_count,total_views
0,GB,QuietChallenge75,208,601205990
1,GB,WildUnboxing39,23,451867023
2,GB,PerfectTrailer65,27,390031997
3,IN,QuietChallenge75,139,388305518
4,GB,FinalSpeech81,15,381963541
5,US,QuietChallenge75,126,350280531
6,US,PerfectTutorial78,38,237932199
7,GB,QuietInterview9,47,216457109
8,GB,PerfectReaction36,25,203916552
9,US,SurprisingHighlights41,42,195881648
